# Committor of a 2D Toytis Staple Run (Brownian or Langevin)

Fits the committor (`CommittorNE`, NPNEq) to the paths of a toytis staple (i\*) run on a 2D potential, and checks it against references. Generic: the run's dynamics (Brownian or Langevin, friction, time step, temperature), interfaces, recorded velocities and potential are read from its `logging.log`; nothing in the code is specific to one potential.

**Path data** (`tis_data.prepare_data`, notes §3–§9): every path is cut at its first frame past its ensemble's interface M, its mirrored piece (the part before the last M crossing, read backwards; velocities negated) is added with `REVERSED`, and every piece carries its path's rejection weight. A and B ends are treated as traps for the metrics.

**CVs** (`CVS = 'auto'`): the positions (y, x) for Brownian runs; for Langevin runs also the recorded velocities (v_y, v_x) if the run saved them (`v_ord = True`), or finite-difference velocities with `FD_VELOCITY`.

**Fit lag** (`FIT_LAG = 'auto'`): 1 for Brownian runs; for Langevin Δ ≈ 0.05/(γ dt) frames with the stopped-window lag-Δ update (notes §10.1), since over one step the energy hardly changes.

**References:**
- *Exact* (Brownian runs): the overdamped committor on a grid (`toy2d.exact_committor_brownian`), absorption every step as in the run. It also tests the data: put into the stopped Z_q of the data, the exact committor should pass at every lag.
- *Shooting* (both dynamics; Langevin needs recorded velocities): the committor of `N_SHOOT_POINTS` random frames from `N_SHOTS` new trajectories each, with the run's integrator.
- The model-free MD reference of `toy2d_md.ipynb` can also be compared (`MD_COMMITTOR_NPZ`).

All figures are shown and saved to `FIG_DIR` (`figures/nb_toy2d_staple/<run>/`); the fit's plots go to `FIG_DIR/<fit tag>/seed<s>/it<N>_<plot>.png`.

## 1. Imports

In [ ]:
import argparse
import os
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
import cupy as cp

plt.rc('font', size=10)
plt.rc('axes', titlesize=12)
plt.rc('axes', labelsize=12)
plt.rc('legend', fontsize=9)

REPO = '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/OP_improvement/optimalrcs'
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs
import seed_runs
import tis_data
import toy2d

## 2. Settings

| Name | Default | Meaning |
|---|---|---|
| `TIS_DIR` | | the staple run |
| `POTENTIAL`, `POT_KWARGS` | `None` | `None`: rebuilt from the run's log; else `module:Class` and its arguments |
| `LAMBDA_A`, `LAMBDA_B` | `None` | states; `None`: first and last interface |
| `PATHS_PER_ENSEMBLE`, `PATH_SEED` | `None`, 0 | random subset of accepted paths per ensemble (`None`: all) |
| `REVERSED`, `WEIGHTS`, `MAX_LENGTH` | True, `rejections`, `None` | mirrored pieces, MC weights, piece-length limit (truncation) |
| `CVS`, `FD_VELOCITY` | `auto`, False | fit CVs (names from y, x, vy, vx, dy, dx); finite-difference velocities |
| `FIT_LAG` | `auto` | fit lag in frames |
| `FIT_SEEDS`, `WARMUP_ITER`, `WARMUP_ENVELOPE` | [0], 1000, 1 | seeds; warm-up with a wide envelope |
| `MAX_ITER`, `CHUNK`, `ENVELOPE`, `FIT_GAMMA`, `NY` | 10000, 2500, 0.01, 0.1, 6 | main fit; plots every chunk |
| `N_SHOOT_POINTS`, `N_SHOTS` | 300, 200 | shooting reference (0: off) |
| `MD_COMMITTOR_NPZ` | `None` | `committor.npz` of `toy2d_md.ipynb` for the same potential and dynamics (optional) |

Rough cost: about 0.4 s per iteration per 20M frames; the stopped-Z_q metrics need about 250 bytes per frame on the GPU, so use `PATHS_PER_ENSEMBLE` to stay well below ~25M frames on an 8 GB GPU.

In [ ]:
TIS_DIR = f'{toy2d.TOYTIS}/simulations/sim_istar_cc_br'
POTENTIAL, POT_KWARGS = None, None
LAMBDA_A, LAMBDA_B = None, None
PATHS_PER_ENSEMBLE, PATH_SEED = None, 0
REVERSED, WEIGHTS, MAX_LENGTH = True, 'rejections', None
CVS, FD_VELOCITY = 'auto', False
FIT_LAG = 'auto'
FIT_SEEDS = [0]
WARMUP_ITER, WARMUP_ENVELOPE = 1000, 1.0
MAX_ITER, CHUNK = 10000, 2500
ENVELOPE, FIT_GAMMA, NY = 0.01, 0.1, 6
N_SHOOT_POINTS, N_SHOTS = 300, 200
MD_COMMITTOR_NPZ = None
MIN_COUNT = 20                      # frames a bin needs to be shown

RUN = os.path.basename(TIS_DIR.rstrip('/'))
run = toy2d.read_full_settings(TIS_DIR)
BROWNIAN = bool(run.get('high_friction', False))
DT, KT, GAMMA, MASS = run['dt'], run['temperature'], run['friction'], run.get('mass', 1.0)
V_ORD = bool(run.get('v_ord', False))
pot = toy2d.potential_from_settings(run, POTENTIAL, POT_KWARGS)
print(f'{RUN}: {"Brownian" if BROWNIAN else "Langevin"}, dt {DT}, kT {KT}, gamma {GAMMA}, mass {MASS}, '
      f'velocities recorded: {V_ORD}, {run["max_cycles"]} cycles requested')
print('interfaces', run['interfaces'])
print('potential', repr(pot))

if CVS == 'auto':
    CVS = ('y', 'x') if BROWNIAN else (('y', 'x', 'vy', 'vx') if V_ORD else ('y', 'x', 'dy', 'dx'))
FD_VELOCITY = FD_VELOCITY or any(c in CVS for c in ('dy', 'dx'))
if FIT_LAG == 'auto':
    FIT_LAG = 1 if BROWNIAN else max(1, int(round(0.05 / (GAMMA * DT))))
fit_tag = (f'{"rev" if REVERSED else "fwd"}_{WEIGHTS}_p{PATHS_PER_ENSEMBLE}_s{PATH_SEED}'
           + (f'_len{MAX_LENGTH}' if MAX_LENGTH else '') + f'_{"-".join(CVS)}_lag{FIT_LAG}'
           + f'_env{ENVELOPE:g}_g{FIT_GAMMA:g}_it{MAX_ITER}')
FIG_DIR = f'{REPO}/examples/figures/nb_toy2d_staple/{RUN}'
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name, fig=None):
    (fig or plt.gcf()).savefig(f'{FIG_DIR}/{name}.png', dpi=110, bbox_inches='tight')

print('CVs', CVS, '| fit lag', FIT_LAG, '| figures', FIG_DIR, '| fit tag', fit_tag)
# === SETTINGS END ===

## 3. Load the Paths

`tis_data.prepare_data` with the settings above; `data.X_all` holds every column of `order.txt` after the time (y, x, and v_y, v_x if recorded), plus Δy, Δx per frame with `FD_VELOCITY`.

In [ ]:
parser = argparse.ArgumentParser()
tis_data.add_data_arguments(parser)
argv = [TIS_DIR, '--weights', WEIGHTS]
if LAMBDA_A is not None:
    argv += ['--lambda-a', str(LAMBDA_A)]
if LAMBDA_B is not None:
    argv += ['--lambda-b', str(LAMBDA_B)]
if PATHS_PER_ENSEMBLE is not None:
    argv += ['--paths-per-ensemble', str(PATHS_PER_ENSEMBLE), '--seed', str(PATH_SEED)]
if REVERSED:
    argv += ['--reversed']
if MAX_LENGTH is not None:
    argv += ['--max-length', str(MAX_LENGTH)]
if FD_VELOCITY:
    argv += ['--fd-velocity']
args = parser.parse_args(argv)
data = tis_data.prepare_data(args)
LAMBDA_A, LAMBDA_B = args.lambda_a, args.lambda_b

cols = {'y': data.X_all[:, 0], 'x': data.X_all[:, 1]}
if V_ORD:
    cols.update(vy=data.X_all[:, 2], vx=data.X_all[:, 3])
if FD_VELOCITY:
    cols.update(dy=data.X_all[:, -2] / DT, dx=data.X_all[:, -1] / DT)      # per-frame change / dt
Y, X = cols['y'], cols['x']
W = data.path_weights
A, B = data.boundary0, data.boundary1
b = A | B
free = ~b
n_pieces, n_paths = len(np.unique(data.i_traj)), len(np.unique(data.group_traj))
print(f'{len(Y):,} frames in {n_pieces} pieces from {n_paths} paths; {A.sum()} frames in A, {B.sum()} in B')

## 4. The Data

- The weighted density of frames on the potential, with the interfaces (it is the path-sampling density, not the equilibrium one).
- Piece lengths and the number of pieces and their total weight per ensemble.
- A few pieces of every ensemble on the potential (forward pieces solid, mirrored dashed).
- Langevin with recorded velocities: their distribution (not Maxwellian in general: the paths are selected).

In [ ]:
y_lo, y_hi = np.percentile(Y, [0.1, 99.9])
x_lo, x_hi = np.percentile(X, [0.1, 99.9])
pad_y, pad_x = 0.05 * (y_hi - y_lo), 0.1 * (x_hi - x_lo)
Y_RANGE, X_RANGE = (y_lo - pad_y, y_hi + pad_y), (x_lo - pad_x, x_hi + pad_x)
gy, gx = np.linspace(*Y_RANGE, 241), np.linspace(*X_RANGE, 161)
V = toy2d.potential_grid(pot, gy, gx)
V_kT = (V - V.min()) / KT
LEVELS = np.arange(0, 12.5, 1.0)
CONTOURS = (gy, gx, V_kT, LEVELS)
INTFS = run['interfaces']

def draw_potential(ax, intfs=True):
    ax.contour(gy, gx, V_kT.T, levels=LEVELS, colors='0.55', linewidths=0.5)
    if intfs:
        for lam in INTFS:
            ax.axvline(lam, color='k', lw=0.5, ls=':')
        for lam in (LAMBDA_A, LAMBDA_B):
            ax.axvline(lam, color='k', lw=1)
    ax.set(xlim=Y_RANGE, ylim=X_RANGE, xlabel='$y$ (order parameter)', ylabel='$x$')

ey, ex = np.linspace(*Y_RANGE, 121), np.linspace(*X_RANGE, 81)
Hw = np.histogram2d(Y, X, [ey, ex], weights=W)[0]
fig, ax = plt.subplots(figsize=(11, 4.8))
im = ax.pcolormesh(ey, ex, np.where(Hw > 0, Hw, np.nan).T, norm='log', cmap='viridis')
draw_potential(ax)
fig.colorbar(im, ax=ax, label='weighted frames')
ax.set_title(f'{RUN}: weighted frame density (dotted: interfaces)')
save_fig('data_density')
plt.show()

ends = np.r_[np.flatnonzero(np.diff(data.i_traj) != 0), len(Y) - 1]
starts = np.r_[0, ends[:-1] + 1]
lengths = ends - starts + 1
piece_lam0 = Y[starts]
ens_of_piece = np.digitize(piece_lam0, INTFS)                 # rough ensemble label: interface window of the start
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.hist(lengths, bins=np.logspace(0, np.log10(lengths.max() + 1), 60))
ax1.set(xscale='log', yscale='log', xlabel='piece length (frames)', ylabel='pieces',
        title=f'median {np.median(lengths):.0f}, 99% {np.percentile(lengths, 99):.0f}, max {lengths.max()}')
piece_w = W[starts]
cnt = np.bincount(ens_of_piece, minlength=len(INTFS) + 1)
wsum = np.bincount(ens_of_piece, weights=piece_w, minlength=len(INTFS) + 1)
ax2.bar(np.arange(len(cnt)) - 0.2, cnt, 0.4, label='pieces')
ax2.bar(np.arange(len(cnt)) + 0.2, wsum, 0.4, label='total weight')
ax2.set(xlabel='interface window of the piece start', ylabel='count / weight')
ax2.legend()
fig.tight_layout()
save_fig('data_pieces')
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.8))
draw_potential(ax)
rng = np.random.default_rng(0)
for k in np.unique(ens_of_piece):
    sel = np.flatnonzero(ens_of_piece == k)
    for i in rng.choice(sel, size=min(3, sel.size), replace=False):
        s0, s1 = starts[i], ends[i] + 1
        ax.plot(Y[s0:s1], X[s0:s1], lw=0.6, color=f'C{k % 10}')
ax.set_title('three random pieces per interface window')
save_fig('data_example_pieces')
plt.show()

if V_ORD and not BROWNIAN:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    vg = np.linspace(-5, 5, 200) * np.sqrt(KT / MASS)
    for ax, name in zip(axes, ('vy', 'vx')):
        ax.hist(cols[name][free][::5], bins=100, density=True, weights=W[free][::5], alpha=0.6, label='frames (weighted)')
        ax.plot(vg, np.exp(-MASS * vg ** 2 / (2 * KT)) / np.sqrt(2 * np.pi * KT / MASS), 'k--', label='Maxwell')
        ax.set(xlabel=name)
        ax.legend()
    fig.tight_layout()
    save_fig('data_velocities')
    plt.show()

## 5. References and the Data Test

For Brownian runs: the exact committor on a grid, interpolated to every frame, and the **data test**: the worst |stopped Z_q / standard error| of the exact committor on these data, per lag (it should be ≲ 2–3 at every lag; a failure means the data, not the fit, are off). For both dynamics: the shooting reference at random frames (Langevin needs recorded velocities; for Brownian runs it also checks the grid solution).

In [ ]:
LAGS = (1, 4, 16, 64, 256, 1024)
ref = q_exact = None
if BROWNIAN:
    ref = toy2d.exact_committor_brownian(pot, KT, GAMMA, DT, LAMBDA_A, LAMBDA_B, stride=1, mass=MASS,
                                         y_range=Y_RANGE, x_range=X_RANGE)
    q_exact = toy2d.interp_grid(ref, Y, X)
    q_exact[A], q_exact[B] = 0.0, 1.0
    z_exact = toy2d.z_per_lag(q_exact, b, data.i_traj, W, LAGS, data.group_traj)
    print('data test, exact committor: worst |z| at lags', LAGS, ':', ' '.join(f'{z:.1f}' for z in z_exact))
    fig, ax = plt.subplots(figsize=(11, 4.8))
    im = ax.pcolormesh(ref['y'], ref['x'], ref['q'].T, cmap='coolwarm', vmin=0, vmax=1)
    ax.contour(ref['y'], ref['x'], ref['q'].T, levels=[0.1, 0.25, 0.5, 0.75, 0.9], colors='k', linestyles='--', linewidths=1)
    draw_potential(ax)
    fig.colorbar(im, ax=ax, label='$q$')
    ax.set_title(f'exact committor (overdamped, gamma {GAMMA}, absorption every step); TPT: {ref["steps_per_ab"]:.3g} steps per A$\\to$B')
    save_fig('exact_committor')
    plt.show()

shoot_idx = np.array([], int)
can_shoot = N_SHOOT_POINTS > 0 and (BROWNIAN or V_ORD) and hasattr(pot, 'potential_and_force_xy')
if can_shoot:
    rng = np.random.default_rng(1)
    shoot_idx = np.sort(rng.choice(np.flatnonzero(free), size=min(N_SHOOT_POINTS, free.sum()), replace=False))
    vy0 = cols['vy'][shoot_idx] if V_ORD else None
    vx0 = cols['vx'][shoot_idx] if V_ORD else None
    t0 = time.time()
    q_shoot, se_shoot = toy2d.shoot(pot, Y[shoot_idx], X[shoot_idx], vy0, vx0, N_SHOTS, DT, KT, GAMMA,
                                    LAMBDA_A, LAMBDA_B, MASS, BROWNIAN, xp=cp, progress=False)
    print(f'shooting: {len(shoot_idx)} frames x {N_SHOTS} trajectories in {time.time() - t0:.0f} s, '
          f'mean standard error {np.nanmean(se_shoot):.3f}')
    if q_exact is not None:
        d = q_exact[shoot_idx] - q_shoot
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.errorbar(q_shoot, q_exact[shoot_idx], xerr=se_shoot, fmt='.', ms=3, alpha=0.6)
        ax.plot([0, 1], [0, 1], 'k--')
        ax.set(xlabel='shooting', ylabel='exact (grid)', title=f'grid vs shooting: RMS {np.sqrt(np.mean(d ** 2)):.3f}')
        save_fig('exact_vs_shooting')
        plt.show()
else:
    print('no shooting reference (needs a vectorized potential, and recorded velocities for Langevin runs)')

md_ref = None
if MD_COMMITTOR_NPZ is not None:
    md_ref = np.load(MD_COMMITTOR_NPZ)
    print('MD reference', MD_COMMITTOR_NPZ, '| settings', str(md_ref['settings']))

## 6. Fit the Committor

`CommittorNE` with the path weights, `group_traj` (forward and mirrored piece of a path as one unit), A/B ends as traps for the metrics, and the update of `FIT_LAG` (`toy2d.lag_update`). `seed_runs.run_seeds` shows and saves, after the warm-up and every `CHUNK` iterations: `metrics`, `tis` (stopped Z_q per lag, short-lag Z_q, stopped obs vs pred), `feps`, `obs_pred`, `obs_pred_log` (the last three are the original, unstopped versions, biased for path pieces), and the fitted q against the CVs (with the exact committor for Brownian runs). Convergence: `zqs_z` ≲ 2 (judge a lag-Δ fit at lags ≥ Δ).

In [ ]:
X_cv = cp.asarray(np.column_stack([cols[c] for c in CVS]))
e_y = np.linspace(LAMBDA_A, LAMBDA_B, 71)
e_x = np.linspace(*np.percentile(X[free], [0.5, 99.5]), 41)
planes = [seed_runs.plane(Y, e_y, '$y$', X, e_x, '$x$', CONTOURS)]
vel = [c for c in CVS if c in ('vy', 'vx', 'dy', 'dx')]
for c in vel:
    lim = np.percentile(np.abs(cols[c][free]), 99)
    planes.append(seed_runs.plane(Y, e_y, '$y$', cols[c], np.linspace(-lim, lim, 41), c))

def make_q():
    q = optimalrcs.CommittorNE(boundary0=A, boundary1=B, i_traj=data.i_traj, t_traj=data.t_traj,
                               path_weights=W, group_traj=data.group_traj)
    q.set_fixed_traj_length_trap(b, 2 ** 16)
    return q

extra_kw = toy2d.lag_update(optimalrcs, data.i_traj, b, W, FIT_LAG)
PRINT_STEP = min(500, CHUNK, WARMUP_ITER or CHUNK)
fit_kw = dict(ny=NY, print_step=PRINT_STEP, save_min_metric='max_z_zq_stopped',
              metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'),
              **extra_kw)
METRICS = ['delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x']
mid_y = (e_y[1:] + e_y[:-1]) / 2
prof_ref = None if q_exact is None else seed_runs.BinnedMean((Y,), (e_y,), free, MIN_COUNT)(q_exact)
extras = {'q_map': seed_runs.committor_map(planes[:1], mask=free, q_ref=q_exact, min_count=MIN_COUNT),
          'q_profile': seed_runs.committor_profile(Y, e_y, '$y$', mask=free,
                                                   ref=None if prof_ref is None else (mid_y, prof_ref))}
if len(planes) > 1:
    extras['q_velocity_planes'] = seed_runs.committor_map(planes[1:], mask=free, min_count=MIN_COUNT)

def report(q):
    r = cp.asnumpy(q.r_traj)
    h = q.metrics_history.get('max_z_zq_stopped', [np.nan])
    out = f'| zqs_z {h[-1]:.2f}'
    if q_exact is not None:
        bm = seed_runs.BinnedMean(planes[0]['c'], planes[0]['edges'], free, MIN_COUNT)
        Qf, Qe = bm(r), bm(q_exact)
        t = (Qe > 0.05) & (Qe < 0.95)
        out += f' | RMS(fit - exact) per bin {np.sqrt(np.nanmean((Qf - Qe)[t] ** 2)):.4f}'
    if shoot_idx.size:
        out += f' | vs shooting {np.sqrt(np.nanmean((r[shoot_idx] - q_shoot) ** 2)):.4f}'
    return out

results, q = seed_runs.run_seeds(
    optimalrcs, make_q,
    comp_y=lambda: X_cv[:, np.random.randint(X_cv.shape[1])],
    fit_kw=fit_kw, seeds=FIT_SEEDS, max_iter=MAX_ITER, chunk=CHUNK, out_dir=f'{FIG_DIR}/{fit_tag}',
    envelope=ENVELOPE, gamma=FIT_GAMMA,
    warmup_iter=WARMUP_ITER, warmup_envelope=WARMUP_ENVELOPE, warmup_gamma=FIT_GAMMA,
    plots=('metrics', 'tis', 'feps', 'obs_pred', 'obs_pred_log'), plot_options=dict(metrics=METRICS),
    extra_plot=extras, report=report,
)
r = cp.asnumpy(q.r_traj)

## 7. Results

- The worst stopped z per lag of the fit (and of the exact committor on the same data).
- q against the order parameter: every frame (sample) and the weighted mean per bin, with the exact committor.
- Maps: fit, exact and fit − exact on (y, x); for velocity CVs the fit on (y, v).
- Isocommittor lines of the fit (and the exact committor) on the potential.
- Fit against the shooting reference, and against the MD committor of `toy2d_md.ipynb` if given.
- Seeds: the fits of all `FIT_SEEDS` against each other (if more than one).

In [ ]:
z_fit = toy2d.z_per_lag(r, b, data.i_traj, W, LAGS, data.group_traj)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(LAGS, z_fit, 'o-', label='fit')
if q_exact is not None:
    ax.plot(LAGS, z_exact, 's--', label='exact committor')
ax.axhline(2, color='k', ls=':')
if FIT_LAG > 1:
    ax.axvline(FIT_LAG, color='C3', lw=0.8, label=f'fit lag {FIT_LAG}')
ax.set(xscale='log', xlabel='lag (frames)', ylabel='worst |stopped $Z_q$ / s.e.|', title='stopped Z_q test per lag (own data)')
ax.legend()
ax.grid(alpha=0.3)
save_fig(f'z_per_lag_{fit_tag}')
plt.show()
print('worst |z| per lag', LAGS, ':', ' '.join(f'{z:.1f}' for z in z_fit))

fig, ax = plt.subplots(figsize=(11, 4.5))
show = np.random.default_rng(0).choice(len(r), size=min(200_000, len(r)), replace=False)
ax.scatter(Y[show], r[show], s=1, alpha=0.15, edgecolors='none', label='frames')
bmw = seed_runs.BinnedMean((Y,), (e_y,), free, MIN_COUNT)
ax.plot(mid_y, bmw(W * r) / bmw(W), 'C3', lw=2, label='weighted mean per bin')
if prof_ref is not None:
    ax.plot(mid_y, bmw(W * q_exact) / bmw(W), 'k--', lw=1.5, label='exact, same frames')
for lam in INTFS:
    ax.axvline(lam, color='k', lw=0.5, ls=':')
ax.set(xlabel='$y$', ylabel='$q$')
ax.legend()
save_fig(f'q_vs_order_{fit_tag}')
plt.show()

fig = seed_runs.committor_map(planes[:1], mask=free, q_ref=q_exact, min_count=MIN_COUNT)(q)
save_fig(f'result_map_{fit_tag}', fig)
plt.show()
if len(planes) > 1:
    fig = seed_runs.committor_map(planes[1:], mask=free, min_count=MIN_COUNT)(q)
    save_fig(f'result_velocity_planes_{fit_tag}', fig)
    plt.show()

p = planes[0]
bm5 = seed_runs.BinnedMean(p['c'], p['edges'], free, 5)
(e1, e2) = p['edges']
m1, m2 = (e1[1:] + e1[:-1]) / 2, (e2[1:] + e2[:-1]) / 2
levels = [0.1, 0.25, 0.5, 0.75, 0.9]
fig, ax = plt.subplots(figsize=(11, 4.8))
draw_potential(ax)
cs = ax.contour(m1, m2, bm5(r).T, levels=levels, colors='C3', linewidths=1.8)
ax.clabel(cs, fmt='%.2f', fontsize=8)
ax.plot([], [], 'C3', label='fit (mean per bin)')
if ref is not None:
    ax.contour(ref['y'], ref['x'], ref['q'].T, levels=levels, colors='C0', linewidths=1.2, linestyles='--')
    ax.plot([], [], 'C0--', label='exact')
ax.legend()
ax.set_title('isocommittor lines 0.1, 0.25, 0.5, 0.75, 0.9 (grey: $V/kT$ = 0, 1, 2, ...)')
save_fig(f'isocommittor_{fit_tag}')
plt.show()

if shoot_idx.size:
    d = r[shoot_idx] - q_shoot
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.errorbar(q_shoot, r[shoot_idx], xerr=se_shoot, fmt='.', ms=3, alpha=0.6)
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set(xlabel='shooting', ylabel='fit',
           title=f'fit vs shooting at {len(shoot_idx)} frames: RMS {np.sqrt(np.nanmean(d ** 2)):.3f}\n'
                 f'mean {np.nanmean(d):+.4f} (shooting error {np.nanmean(se_shoot):.3f})')
    save_fig(f'fit_vs_shooting_{fit_tag}')
    plt.show()

if md_ref is not None:
    # the MD fit, binned on the same (y, x) bins as this fit (mean over the MD frames vs over the path frames)
    bm_md = seed_runs.BinnedMean((md_ref['y'], md_ref['x']), p['edges'], None, MIN_COUNT)
    bm_pf = seed_runs.BinnedMean(p['c'], p['edges'], free, MIN_COUNT)
    Qm, Qp = bm_md(md_ref['q_fit']), bm_pf(r)
    fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
    for ax, (M, title, cmap, lim) in zip(axes, [(Qp, 'staple fit', 'coolwarm', (0, 1)), (Qm, 'MD fit', 'coolwarm', (0, 1)),
                                               (Qp - Qm, 'staple $-$ MD (bin means)', 'PuOr', (-0.15, 0.15))]):
        im = ax.pcolormesh(e1, e2, M.T, cmap=cmap, vmin=lim[0], vmax=lim[1])
        ax.set(title=title, xlabel='$y$', ylabel='$x$')
        fig.colorbar(im, ax=ax)
    fig.suptitle('Note: for Langevin dynamics the bin means average over different velocity distributions')
    fig.tight_layout()
    save_fig(f'staple_vs_md_{fit_tag}', fig)
    plt.show()

if len(results) > 1:
    seeds = list(results)
    fig, ax = plt.subplots(figsize=(5.5, 5))
    for s in seeds[1:]:
        ax.plot(results[seeds[0]]['r'][free][::200], results[s]['r'][free][::200], '.', ms=2, label=f'seed {s}')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set(xlabel=f'seed {seeds[0]}', ylabel='other seeds', title='fits with different seeds (every 200th frame)')
    ax.legend()
    save_fig(f'seeds_{fit_tag}')
    plt.show()

## 8. Save

The fitted q per frame with the frames, pieces, weights and references, as `FIG_DIR/committor_<fit tag>.npz`.

In [ ]:
out = f'{FIG_DIR}/committor_{fit_tag}.npz'
np.savez(out, q_fit=r.astype(np.float32), X_all=data.X_all.astype(np.float32), i_traj=data.i_traj,
         group_traj=data.group_traj, t_traj=data.t_traj, path_weights=W, boundary0=A, boundary1=B,
         q_exact=(q_exact if q_exact is not None else np.full(len(r), np.nan)).astype(np.float32),
         shoot_idx=shoot_idx, q_shoot=q_shoot if shoot_idx.size else [], z_fit=z_fit, lags=LAGS,
         settings=repr(dict(run=RUN, cvs=CVS, fit_lag=FIT_LAG, reversed=REVERSED, weights=WEIGHTS,
                            paths_per_ensemble=PATHS_PER_ENSEMBLE, path_seed=PATH_SEED, max_length=MAX_LENGTH,
                            envelope=ENVELOPE, fit_gamma=FIT_GAMMA, ny=NY, max_iter=MAX_ITER, warmup=WARMUP_ITER,
                            seeds=FIT_SEEDS)))
print('saved', out)